**Assignment 3: Introduction to Spark Structured Streaming**\
**25MML1001 Karthik M**

# Theory

Q. What is Structured Streaming in Spark? Explain how it differs from batch processing. Discuss its key features and real-time use cases.

## What is Structured Streaming?

**Structured Streaming** is Spark's engine for real-time data processing. The key idea is elegant — treat a live data stream as an **unbounded table that keeps growing**. Every new batch of incoming data is like new rows appended to this table, and you write the same DataFrame/SQL queries you would use for batch data.

This means there is no separate streaming API to learn — the same `filter()`, `groupBy()`, `select()` calls work on both static and streaming DataFrames.

---

## Batch Processing vs Structured Streaming

| Feature | Batch Processing | Structured Streaming |
|---|---|---|
| Data | Fixed, stored dataset | Continuous, live data |
| Trigger | Manually run once | Triggered by new data arrival |
| Latency | Minutes to hours | Milliseconds to seconds |
| Output | Written once at the end | Continuously updated |
| Fault tolerance | Re-run the entire job | Checkpointing + Write-Ahead Log |
| Use case | Monthly reports, ETL jobs | Fraud detection, live dashboards |
| API | DataFrame / SQL | Same DataFrame / SQL (unified) |

---

## Key Features of Structured Streaming

1. **Exactly-once semantics** – Every record is processed exactly once. No duplicates, no missed records, even if a node fails mid-stream.

2. **Event-time processing** – Spark can process data based on the time the event *actually occurred*, not the time it arrived. This handles late-arriving data gracefully using **watermarks**.

3. **Unified API** – The same DataFrame and SQL API used for batch jobs works for streaming, so there is no extra learning curve.

4. **Multiple output sinks** – Results can be written to the console, files, Kafka, databases, or kept in memory for downstream queries.

5. **Checkpointing** – Spark periodically saves the streaming state to disk so it can recover exactly from where it left off after a failure.

---

## Real-Time Use Cases

- **Fraud detection** – Flag suspicious banking transactions within milliseconds of them occurring
- **Live e-commerce dashboards** – Track order counts, revenue, and inventory in real time
- **Log monitoring** – Detect errors or anomalies in application logs as they stream in (DevOps)
- **Social media trend analysis** – Count hashtags or keywords in live tweet streams
- **IoT sensor monitoring** – Process data from thousands of sensors continuously and trigger alerts

# Practical

Q. Write a PySpark Structured Streaming program to:
- Read data from a streaming source (e.g., socket or file directory)
- Perform a simple transformation (such as word count)
- Output the results to the console
- Explain each step in comments

In [1]:
# Install PySpark
!pip install pyspark -q

In [2]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, col
import os
import time
import threading

In [3]:
# Initiate Spark Session
spark = SparkSession.builder \
    .appName("DA3_StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()

# Suppress verbose Spark logs for cleaner output
spark.sparkContext.setLogLevel("ERROR")

In [4]:
# Create input directory for streaming source
# Spark will watch this folder and process any new .txt files dropped into it
input_dir = "/tmp/streaming_input"
os.makedirs(input_dir, exist_ok=True)
print(f"Streaming input directory created: {input_dir}")

Streaming input directory created: /tmp/streaming_input


In [5]:
# Write sample text files into the input directory
# These simulate incoming streaming data (like log lines or messages)
sample_data = [
    "hello world hello spark",
    "spark is fast spark is powerful",
    "hello from structured streaming",
    "distributed data processing with spark"
]

for i, line in enumerate(sample_data):
    file_path = os.path.join(input_dir, f"file_{i}.txt")
    with open(file_path, "w") as f:
        f.write(line + "\n")

print("Sample streaming files written successfully.")

Sample streaming files written successfully.


In [6]:
# STEP 1: Define the streaming source
# Spark reads new text files added to the input directory as a stream
# Each line in the file becomes one row in the streaming DataFrame
lines = spark.readStream \
    .format("text") \
    .option("path", input_dir) \
    .load()

print("Streaming source defined. Schema:")
lines.printSchema()

Streaming source defined. Schema:
root
 |-- value: string (nullable = true)



In [7]:
# STEP 2: Transformation – Word Count
# split()   → splits each line by space into an array of words
# explode() → converts the array into individual rows (one word per row)
# groupBy() → groups identical words together
# count()   → counts how many times each word appears

words = lines.select(
    explode(
        split(col("value"), " ")
    ).alias("word")
)

word_counts = words.groupBy("word").count().orderBy("count", ascending=False)

In [8]:
# STEP 3: Define the output sink
# format("console") → prints results to the console after every trigger
# outputMode("complete") → prints the full updated word count table each time
# trigger → runs the streaming query every 5 seconds
# truncate=False → shows full word text without cutting it off

query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Streaming query started. Waiting for results...")

Streaming query started. Waiting for results...


In [9]:
# STEP 4: Let the query run for 15 seconds to process all files
# then stop it. In a real system, awaitTermination() would run forever.
query.awaitTermination(timeout=15)
print("Streaming query stopped.")

Streaming query stopped.


In [10]:
# End Spark Session
spark.stop()